# 03 · 包、模块与命令行 Packaging & CLI

> **能力标签**：SH1（Python 编程能力）、SH2（工具与可复现性）

## 目标 Objectives

完成本课后，你应该能够：

1. 区分**模块（module）**和**包（package）**，理解 `__init__.py` 的作用。
2. 用绝对导入和相对导入引用自定义模块中的函数和类。
3. 用 `pathlib.Path` 操作文件路径（跨平台、面向对象风格）。
4. 用 `argparse` 解析命令行参数：位置参数、可选参数（`--name`）、布尔标志（`--verbose`）、默认值。
5. 用 `python -m <package>` 的方式运行包，理解 `__main__.py` 的作用。

> 对应能力：**SH1, SH2**


## 概念讲解 Concepts

### 模块 vs 包

- **模块（module）**：一个 `.py` 文件就是一个模块。`import math` 导入的就是标准库里的 `math.py`。
- **包（package）**：一个含有 `__init__.py` 的目录就是一个包。

```
myproject/
├── __init__.py        ← 有这个，myproject 就是一个包
├── utils.py           ← myproject.utils 模块
└── cli/
    ├── __init__.py    ← cli 是 myproject 的子包
    └── parser.py      ← myproject.cli.parser 模块
```

---

### `__init__.py`

- 空文件也行，作用是告诉 Python"这个目录是包"。
- 可以在里面做**公开接口（public API）**的汇总：

```python
# myproject/__init__.py
from .utils import helper_fn   # 让 from myproject import helper_fn 生效
```

---

### 导入（Import）

```python
import math                      # 导入整个模块
from math import sqrt, pi        # 只导入需要的名字
from math import sqrt as sq      # 起别名

# 相对导入（只在包内部使用）
from .utils import helper_fn     # 同级模块
from ..core import Model         # 上一级的 core 模块
```

> **原则**：在包内部用相对导入；在脚本或测试中用绝对导入。

---

### `pathlib.Path`

比 `os.path` 更现代、更易读的路径操作方式：

```python
from pathlib import Path

p = Path("data/raw")
p.mkdir(parents=True, exist_ok=True)   # 创建目录（包括父目录）

csv_files = list(p.glob("*.csv"))      # 列出所有 CSV 文件

for f in csv_files:
    print(f.stem, f.suffix, f.parent)  # 文件名(无后缀)、后缀、父目录
```

路径可以用 `/` 拼接：`p / "subdir" / "file.txt"`

---

### `argparse`：命令行参数解析

`argparse` 是标准库中解析命令行参数的工具：

```python
import argparse

def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="我的工具")

    # 位置参数（positional）：必须提供
    p.add_argument("input", help="输入文件路径")

    # 可选参数（optional）：--name 形式，有默认值
    p.add_argument("--n",       type=int,   default=1,       help="重复次数")
    p.add_argument("--name",    type=str,   default="world", help="名字")

    # 布尔标志（flag）：有则为 True，无则为 False
    p.add_argument("--verbose", action="store_true",         help="详细输出")

    return p
```

**在 notebook 里测试**（不真正调用命令行）：

```python
args = p.parse_args(["--n", "3", "--verbose"])
print(args.n, args.verbose)   # 3  True
```

---

### `python -m <package>` 与 `__main__.py`

在包里创建 `__main__.py`，就可以用 `python -m mypackage` 运行它：

```
mypackage/
├── __init__.py
├── __main__.py    ← python -m mypackage 会执行这个
└── core.py
```

`__main__.py` 通常只写一行入口：

```python
from .cli import main
main()
```

这也是为什么 `python -m pytest` 能直接跑测试的原因。


## 示例 Worked Example

下面演示 `w1-cli` 练习包里的核心模式：`build_parser()` + `parse(argv)` 函数。
完全自包含，不需要创建实际文件。


In [ ]:
from __future__ import annotations
import argparse


def build_parser() -> argparse.ArgumentParser:
    """构建命令行参数解析器。"""
    p = argparse.ArgumentParser(description="w1-cli 演示工具")
    p.add_argument("--n",       type=int,   default=1,       help="重复次数")
    p.add_argument("--name",    type=str,   default="world", help="目标名字")
    p.add_argument("--verbose", action="store_true",         help="详细模式")
    return p


def parse(argv: list[str]) -> dict:
    """解析参数列表，返回字典形式的结果。"""
    a = build_parser().parse_args(argv)
    return {"n": a.n, "name": a.name, "verbose": a.verbose}


# ── 在 notebook 里调用（不需要命令行）────────────────────────────────────
result = parse(["--n", "3", "--name", "Alice", "--verbose"])
print("解析结果：", result)

result2 = parse([])   # 全部使用默认值
print("默认值：  ", result2)


In [ ]:
# 演示 pathlib.Path 的常用操作（全部在内存里，不实际创建文件）
from pathlib import Path

# 路径拼接
base = Path("/data/project")
data_dir = base / "raw" / "2024"
print("完整路径：", data_dir)
print("文件名：  ", data_dir.name)
print("父目录：  ", data_dir.parent)
print("各部分：  ", data_dir.parts)

# 模拟文件名处理
sample = Path("results/experiment_01.csv")
print()
print("stem   =", sample.stem)     # 'experiment_01'
print("suffix =", sample.suffix)   # '.csv'
print("name   =", sample.name)     # 'experiment_01.csv'

# 路径判断（Path 对象本身不需要文件真实存在）
print("是绝对路径？", sample.is_absolute())
print("绝对化：    ", sample.resolve())


## 动手 Exercise

在下面的 cell 中，**为 `build_parser_ex()` 添加 `--scale` 参数**：

- 参数名：`--scale`
- 类型：`float`
- 默认值：`1.0`
- 帮助文字："缩放系数"
- 添加完成后，自己加上 `--scale` 后手动测试：`parse_ex(["--scale", "2.5"])` 应返回 `{"n": 1, "name": "world", "verbose": False, "scale": 2.5}`

> **注意**：完成 TODO 注释处的代码后，取消注释验证 cell 中的测试行并手动运行验证。

---

In [ ]:
from __future__ import annotations
import argparse


def build_parser_ex() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="扩展版 CLI")
    p.add_argument("--n",       type=int,   default=1)
    p.add_argument("--name",    type=str,   default="world")
    p.add_argument("--verbose", action="store_true")
    # TODO: 在此添加 --scale 参数：type=float, default=1.0, help="缩放系数"
    return p


def parse_ex(argv: list[str]) -> dict:
    a = build_parser_ex().parse_args(argv)
    # TODO: 在返回字典中加入 "scale": a.scale
    return {"n": a.n, "name": a.name, "verbose": a.verbose}

In [ ]:
# 验证（添加 --scale 后手动在此运行）
# 完成 TODO 后，取消注释并执行：
# r = parse_ex(["--scale", "2.5"])
# print("scale == 2.5 ?", r.get("scale") == 2.5)
# r2 = parse_ex([])
# print("default scale == 1.0 ?", r2.get("scale") == 1.0)

# 先验证基础参数仍然正确：
r_base = parse_ex([])
print("基础参数验证：", {"n": r_base.get("n") == 1, "name": r_base.get("name") == "world", "verbose": r_base.get("verbose") == False})

## 延伸阅读 Further Reading

1. **Python 官方文档 — argparse**：<https://docs.python.org/3/library/argparse.html>
2. **Python 官方文档 — pathlib**：<https://docs.python.org/3/library/pathlib.html>
3. **Python 官方教程 — 模块**：<https://docs.python.org/3/tutorial/modules.html>
4. **`click` 库**：更现代的 CLI 框架，`argparse` 的流行替代品，值得了解。
5. **PEP 328 — 相对导入语法**：<https://peps.python.org/pep-0328/>


## 关联练习 Related Assignments

本课对应练习包：`assessments/autograder/w1-cli`

**运行自动评分器**：

```bash
cd assessments/autograder/w1-cli
python -m pytest test_hidden.py -v
```

或使用平台工具：

```bash
python check.py w1-cli
```

> 目标通过率 ≥ 70%（`threshold_pass: 0.7`）
> 能力标签：**SH1, SH2**
